# The HALD Knowledge Graph

This section has been inspired by the work of Robert Haas on Biomedical Knowledge Graphs which can be found at 
https://github.com/robert-haas/awesome-biomedical-knowledge-graphs/tree/main
The source of the data is the webpage on [Figshare](https://figshare.com/articles/dataset/HALD_a_human_aging_and_longevity_knowledge_graph_for_precision_gerontology_and_geroscience_analyses/22828196). This include versions in JSON and CSV. The CSV versions are structured for a particular package which we will not be using, so we will use the json packages which are more general.

First we'll create a data directory.

In [2]:
import os
Download = False
datadir = "HALD_Dataset"
if not os.path.exists(datadir):
    os.mkdir(datadir)

Now we define the list of the files that we want to download. We'll define a *list* of *tuples*, with each tuple representing one of the files that we want to fetch, specifying three things:

* The name that we want the file to be called.
* The URL from where it will be downloaded.
* The MD5 checksum which will allow us to verify the downloaded file's integrity.

In [3]:
# List of files to download
filelist = [
    ("Entity_info.json", "https://figshare.com/ndownloader/files/43612509", '1746cde24a1bac0460f1ccf646608cc9'),
    ("Literature_Info.json", "https://figshare.com/ndownloader/files/43612512", "10b78e8ec30f5b85f2a58d8fe24f056b"),
    ("Longevity_Biomarkers.json", "https://figshare.com/ndownloader/files/43612497", "0dbd9c3f8474dc3cd744ed38af460d75"),
    ("Relation_Info.json", "https://figshare.com/ndownloader/files/43612506", "0c1fa199269adc58f64ad4d5b9fd87b9"),
    ("Aging_Biomarkers.json", "https://figshare.com/ndownloader/files/43612503", "abd0eb6cb7295ae500c5d676b7797324")
]

Now we can download the files. For each file in `filelist` we will:

* Download the file from the URL.
* If the download request indicates that the download is unsuccessful, print an error.
* If the download is successfull, verify the checksum and if that is correct, write the file to disk in `datadir`

In [4]:
import requests
import hashlib

if Download:
    for f in filelist:
        response = requests.get(f[1])
        file_Path = datadir + "/" + f[0]
        if response.status_code != 200:
            print('Failed to download file {f[0]} from {f[1]}')
        else:
            m = hashlib.md5()
            m.update(response.content)
            if m.hexdigest() == f[2]:
                print(f"SUCCESS: File {f[0]} downloaded from {f[1]} with correct checksum {f[2]}")
                with open(file_Path, 'wb') as file:
                    file.write(response.content)
            else:
                print(f"ERROR: File {f[0]} downloaded from {f[1]} with incorrect checksum {m.hexdigest()} (should be {f[2]})")            


## What does the data look like?

Let us inspect these files. The two key files here are those containing the *entities* (nodes) and the *edges* (relations).

In [5]:
import json

def load_json(fname):
    with open(fname, 'rb') as file:
        return json.load(file)

Entity_info = load_json(f"{datadir}/{filelist[0][0]}")
Literature_info = load_json(f"{datadir}/{filelist[1][0]}")
Longevity_Biomarkers = load_json(f"{datadir}/{filelist[2][0]}")
Relation_info = load_json(f"{datadir}/{filelist[3][0]}")
Aging_Biomarkers = load_json(f"{datadir}/{filelist[4][0]}")

What are these new data structures?

In [6]:
print(type(Entity_info))
print(type(Relation_info))

<class 'dict'>
<class 'dict'>


How big are they?

In [7]:
print(f"There are {len(Entity_info)} Entities and {len(Relation_info)} Relations")


There are 12257 Entities and 116495 Relations


Take a better look at the dataset: what's in it?

In [8]:
# Get the entity keys
first_five_keys = list(Entity_info.keys())[0:5]
print(first_five_keys)
for k in first_five_keys:
    print(f"{k}: {Entity_info[k]}")
    if len(Entity_info[k]) != 1:
        print(f"Entity {k} is a multilist")

['MLH1', 'CD4', 'INS', 'MAPT', 'MYC']
MLH1: [{'entity': 'MLH1', 'type': 'Gene', 'PMID': ['12612901', '30275527', '25311944', '22936446', '19949675', '22406557', '23240038', '11325821', '21042749', '25556597', '17556535', '29425284', '22740444', '10954253', '37380216'], 'official full name': 'mutL homolog 1', 'sentence': [['Most such cancers have the CpG island methylator phenotype (CIMP+) with methylation and transcriptional silencing of the mismatch repair gene MLH1.'], ['Our group recently demonstrated that aging human HSCs accumulate microsatellite instability coincident with loss of MLH1, a DNA Mismatch Repair (MMR) protein, which could reasonably predispose to radiation-induced HSC malignancies.', 'In addition, whole-exome sequencing analysis revealed high SNVs and INDELs in lymphomas being driven by loss of Mlh1 and frequently mutated genes had a strong correlation with human leukemias.'], ['ARID1A loss was observed in 9% (22/257) of the cohort: 24% of MMR-deficient tumors (14/59

So each entry is a list of length 1, and that entry contains a dictionary of the node's attributes. Those attributes are sometimes lists.

Let's get a list of the types of the entities create them. Whilst we are doing this, we rename one of the types as it conflicts with a Python keyword

In [9]:
import re

EntityTypes = set()

    
#entities_with_spaces = [a for a in Entity_info.keys() if ' ' in a]
#print(entities_with_spaces)

# Clean up the name
names_to_be_changed = []
for entity in Entity_info.keys():
    names_to_be_changed.append((entity, re.sub('[^0-9a-zA-Z]+', '_', entity)))

for i in names_to_be_changed:
    Entity_info[i[1]] = Entity_info.pop(i[0],None)

# rename a troublesome entity
for k in Entity_info.keys():
    Entity_info[k][0]['entitytype'] = Entity_info[k][0].pop('type',None)
    EntityTypes.add(Entity_info[k][0]['entitytype'])

from owlready2 import *
onto = get_ontology("http://www.dummy.info/new.owl")

with onto:
    EntityClasses = dict()
    for entity in EntityTypes:
        print(entity)
        EntityClasses[entity] = type(entity, (Thing,), dict())
        print(f'Created entity class {EntityClasses[entity]}')


Gene
Created entity class new.Gene
RNA
Created entity class new.RNA
Pharmaceutical Preparations
Created entity class new.Pharmaceutical Preparations
Protein
Created entity class new.Protein
Toxin
Created entity class new.Toxin
Peptide
Created entity class new.Peptide
Lipid
Created entity class new.Lipid
Mutation
Created entity class new.Mutation
Carbohydrate
Created entity class new.Carbohydrate
Disease
Created entity class new.Disease


Now we do the same for the relationships. We do this now because we will want to attach attributes to them so we need to create all the attributes in one go.

In [10]:
# Remove a trouble relation from the dataset. For reasons unknown, the relationship type "rs1556516" does not play well with owlready2
Relation_info.pop('Non-alcoholic Fatty Liver Disease-rs1556516-CDKN2B-AS1', None)

# First fetch all the types of relation and  create the ObjectProperties
with onto:
    # Deal with some troublesome relationship names

    problem_cases = [relation for relation in Relation_info.keys() if Relation_info[relation]['relationship'] in ['name']]
    for relation in problem_cases:
        Relation_info[relation]['relationship'] = f"xx{Relation_info[relation]['relationship']}"
    

    # Clean up the name
    names_to_be_changed = []
    for relation in Relation_info.keys():
        names_to_be_changed.append((relation, re.sub('[^0-9a-zA-Z]+', '_', relation)))
    for i in names_to_be_changed:
        Relation_info[i[1]] = Relation_info.pop(i[0],None)
    

    RelationType = dict()
    for relation in Relation_info.keys():
        Relation_info[relation]['relationship'] = "_"+re.sub('[^0-9a-zA-Z]+', '_', Relation_info[relation]['relationship'])
        Domain = Relation_info[relation]['source type']
        Range = Relation_info[relation]['target type']
        DomainClasses = {EntityClasses[i] for i in Domain}
        RangeClasses = {EntityClasses[i] for i in Range}
        RelationType[Relation_info[relation]['relationship']] = {'domain': list(DomainClasses), 'range': list(RangeClasses)}
        #print(RelationType[relation])

    RelationClasses = dict()
    for relation in RelationType.keys():
        RelationClasses[relation] = type(relation, (ObjectProperty,), dict())
        #print(f'Created relation class {RelationClasses[relation]} with properties {RelationType[relation]}')

In [11]:
Relation_info.keys()

dict_keys(['Pulmonary_Disease_Chronic_Obstructive_defined_Inflammation', 'Anorexia_associate_Sarcopenia', 'Sarcopenia_associate_Anorexia', 'GPT_recognized_Death', 'GPT_increase_Death', 'Dementia_cope_Vision_Disorders', 'Dementia_include_Endotoxemia', 'Death_develop_Aggressive_Periodontitis', 'CD8A_expanded_Lung_Neoplasms', 'Constipation_associated_Sarcopenia', 'Sarcopenia_associated_Constipation', 'Dementia_commence_Cholesterol', 'Head_and_Neck_Neoplasms_exclude_Melanoma', 'Atrial_Fibrillation_coexist_with_Hypertension', 'Atrial_Fibrillation_multiply_Stroke', 'Atrial_Fibrillation_multiply_Death', 'Atrial_Fibrillation_coexist_Hypertension', 'Hypertension_multiply_Death', 'Communicable_Diseases_infect_Infections', 'HIV_Infections_live_Heart_Diseases', 'HIV_Infections_managed_Chronic_Disease', 'Blindness_noted_Uveitis', 'Blindness_noted_Panuveitis', 'Pituitary_Neoplasms_considered_Diabetes_Mellitus_Type_2', 'Doxorubicin_stratify_Liposarcoma', 'Nervous_System_Diseases_reduce_Diabetes_Melli

Now we have the class for the entity and the relations, we can construct the unified set of attributes. We loop over the entities identifying new attributes and their domain and range. Here we need to be mindful of a limitation: Owlready2 does not support all datatypes as ranges for the DataProperty class. We will ignore those attributes. The list of valid types for the range can be found at https://owlready2.readthedocs.io/en/latest/properties.html

In [12]:
# First do the entities
EntityAttributeType = dict()

for entity in Entity_info.keys():
    # Replace any underscores in attribute names
    #attributes_with_spaces = [a for a in Entity_info[entity][0].keys() if ' ' in a]
    #print(attributes_with_spaces)
    #for attribute in :   
    #    Entity_info[entity][0][attribute.replace(" ","_")] = Entity_info[entity][0].pop(attribute)
    
    # Clean up the 
    names_to_be_changed = []
    for attribute in Entity_info[entity][0].keys():
        names_to_be_changed.append((attribute, "_"+re.sub('[^0-9a-zA-Z]+', '_', attribute)))
    for i in names_to_be_changed:
        Entity_info[entity][0][i[1]] = Entity_info[entity][0].pop(i[0],None)


    for attribute in Entity_info[entity][0].keys():
        ClassOfType = EntityClasses[Entity_info[entity][0]['_entitytype']]
        TypeOfEntityAttribute = type(Entity_info[entity][0][attribute])
        # Can't have list as a type in owl so need to get the type of the list elements instead
        if attribute not in EntityAttributeType.keys():
            EntityAttributeType[attribute] = {'domain':  {ClassOfType}, 'range': {TypeOfEntityAttribute}}
        else:
            EntityAttributeType[attribute]['domain'].add(ClassOfType)
            EntityAttributeType[attribute]['range'].add(TypeOfEntityAttribute)

AttributesToBeRemoved = list()
for attribute in EntityAttributeType.keys():
    EntityAttributeType[attribute]['domain'] = list(EntityAttributeType[attribute]['domain'])
    EntityAttributeType[attribute]['range'] = list(EntityAttributeType[attribute]['range'])
    if any(a not in [str,int,float,bool] for a in EntityAttributeType[attribute]['range']):
           AttributesToBeRemoved.append(attribute)
    print(f"{attribute}: {EntityAttributeType[attribute]}")

for attribute in AttributesToBeRemoved:
    print(f"Removing attribute {attribute} because of incompatible type")
    EntityAttributeType.pop(attribute, None)



_entity: {'domain': [new.Pharmaceutical Preparations, new.Carbohydrate, new.Lipid, new.Protein, new.Disease, new.RNA, new.Mutation, new.Gene, new.Peptide, new.Toxin], 'range': [<class 'str'>]}
_PMID: {'domain': [new.Pharmaceutical Preparations, new.Carbohydrate, new.Lipid, new.Protein, new.Disease, new.RNA, new.Mutation, new.Gene, new.Peptide, new.Toxin], 'range': [<class 'list'>]}
_official_full_name: {'domain': [new.Pharmaceutical Preparations, new.Carbohydrate, new.Lipid, new.Protein, new.Disease, new.RNA, new.Mutation, new.Gene, new.Peptide, new.Toxin], 'range': [<class 'NoneType'>, <class 'str'>]}
_sentence: {'domain': [new.Pharmaceutical Preparations, new.Carbohydrate, new.Lipid, new.Protein, new.Disease, new.RNA, new.Mutation, new.Gene, new.Peptide, new.Toxin], 'range': [<class 'list'>]}
_numbers_of_articles: {'domain': [new.Pharmaceutical Preparations, new.Carbohydrate, new.Lipid, new.Protein, new.Disease, new.RNA, new.Mutation, new.Gene, new.Peptide, new.Toxin], 'range': [<cla

Now we can create the classes for the attributes

In [13]:
EntityAttributeType.pop('type',None)
with onto:
    AttributeClasses = dict()
    print(EntityAttributeType.keys())
    for attribute in EntityAttributeType.keys():
        print(attribute)
        print(EntityAttributeType[attribute])
        AttributeClasses[attribute] = type(attribute, (DataProperty,), EntityAttributeType[attribute])
        print(f'Created attribute class {AttributeClasses[attribute]}')

dict_keys(['_entity', '_numbers_of_articles', '_alias_names', '_description', '_mutation_position', '_mutation_alleles', '_MeSH_ID', '_relation', '_aging_biomarker', '_longevity_biomarker', '_entitytype'])
_entity
{'domain': [new.Pharmaceutical Preparations, new.Carbohydrate, new.Lipid, new.Protein, new.Disease, new.RNA, new.Mutation, new.Gene, new.Peptide, new.Toxin], 'range': [<class 'str'>]}
Created attribute class new._entity
_numbers_of_articles
{'domain': [new.Pharmaceutical Preparations, new.Carbohydrate, new.Lipid, new.Protein, new.Disease, new.RNA, new.Mutation, new.Gene, new.Peptide, new.Toxin], 'range': [<class 'int'>]}
Created attribute class new._numbers_of_articles
_alias_names
{'domain': [new.Pharmaceutical Preparations, new.Carbohydrate, new.Lipid, new.Protein, new.Disease, new.RNA, new.Mutation, new.Gene, new.Peptide, new.Toxin], 'range': [<class 'str'>]}
Created attribute class new._alias_names
_description
{'domain': [new.Pharmaceutical Preparations, new.Carbohydrate

Now we can populate. First we populate the entities. Now to populate the entities. There is one small issue here: there is an entry in the data "entity": "Disease". This conflicts the name of the one of the types, and hence of one of the EntityClasses. We have to trap for it and rename it.

In [14]:
Nodes = dict()
with onto:
    for entity in Entity_info.keys():
        # Remove invalid attributes
        NodeAttributes = Entity_info[entity][0]
        for attribute in AttributesToBeRemoved:
            NodeAttributes.pop(attribute,None)
        NodeName = re.sub('[^0-9a-zA-Z]+', '_', NodeAttributes.pop('_entity', None))
        print(NodeName)
        NodeKey = entity
        if entity == "Disease":
            print(EntityAttributeType.keys())
            print(NodeAttributes)
            NodeName = 'DiseaseNode'
            NodeKey = NodeName
        Nodes[NodeKey] = EntityClasses[NodeAttributes['_entitytype']](name=NodeName)
        for k in NodeAttributes.keys():
            getattr(Nodes[NodeKey],k).append(NodeAttributes[k])

MLH1
CD4
INS
MAPT
MYC
GSR
SOD2
CRP
IL6
SIRT1
CHGA
CFB
SKIV2L
TNXB
FKBPL
NOTCH4
CFH
HTRA1
GCG
IGF1
GH1
GHRH
WRN
NFKB1
SHBG
PIAS4
CCL2
RECQL4
BLM
ALB
TNF
BCAM
CD151
GGH
FGF23
PTH
JUNB
H2AZ1
PAPPA2
ELN
KIT
CSF2
VEGFA
MYO5A
MTOR
KLK3
AR
ACE
LMNB1
LMNA
NUP62
ULK1
MAP1LC3A
PIK3R2
IAPP
VDR
CLPS
APOD
FERMT2
MS4A6A
ABCA7
SORL1
HTT
APOB
RAF1
MAPK3
MAPK1
MAP2K1
MAP2K2
CFI
SERPINA1
IL7
KL
BECN1
NFE2L2
SENP7
MOB1B
CARMIL1
PRRC2A
TERF2
RFWD3
PARP1
POT1
ATM
MPHOSPH6
PPARGC1A
FNDC5
BDNF
NTRK2
CD8A
IFITM3
TRIM22
LY6E
IFNAR1
CTNNB1
APOL1
VWF
ATR
RNF8
BRCA1
TP53BP1
RETN
CXCL8
IL10
IL1B
IL13RA2
CXCR4
POU5F1
NANOG
IL2
APOE
NDRG2
BACE1
GGA3
CDK5
PIN1
STAT3
IFNG
KCNJ10
AGER
AVP
GPT
SLC17A5
NEDD8
ISG15
URM1
ATG12
GABARAPL1
FAU
UCP1
HIF1A
TET2
ASXL1
RUNX1
KMT2A
HLA_DRB1
NR1I3
CCL11
TNFSF13B
IL12B
PLTP
CHIT1
CXCL10
IL1RL2
AKT1
TP53
SOD1
IL1A
IFNA2
SLC6A2
TARDBP
SST
CDH2
SPHK1
ADA
ADAR
ADARB1
ADARB2
ADIPOQ
POMC
HSP90AA1
ERBB2
HLA_DQB1
KLF4
SOX2
CDKN2A
CDKN2B
GPM6A
PAK1
TTR
CLU
TRPM2
XRCC6
XRCC5
DBP
RUNX1T1
MYH11

In [15]:
Nodes['MLH1']._aging_biomarker

[True]

Now add the relations

In [16]:
with onto:
    for relation in Relation_info.keys():
        SourceNodeName = Relation_info[relation]['source entity']
        TargetNodeName = Relation_info[relation]['target entity']
        RelationName = Relation_info[relation]['relationship']
        #print(f"{SourceNodeName} {RelationName} {TargetNodeName}")
        if SourceNodeName in Nodes.keys() and TargetNodeName in Nodes.keys():
            #print(Nodes[SourceNodeName])
            #print(getattr(Nodes[SourceNodeName],RelationName))
            #print(Nodes[TargetNodeName])
            getattr(Nodes[SourceNodeName],RelationName).append(Nodes[TargetNodeName])
        #else:
        #    print(f"{SourceNodeName} {RelationName} {TargetNodeName}")
    

# Now we populate
#with onto:
#    for row in relationstable.iterrows():
#        x = row[1].to_dict()
#        getattr(nodes[x['src']],x['relation']).append(nodes[x['dest']])

In [17]:
RelationType['_xxname']

{'domain': [new.Disease], 'range': [new.Disease]}

In [18]:
Nodes['Immunologic_Deficiency_Syndromes']

new.Immunologic_Deficiency_Syndromes

In [19]:
onto.save('HALD.rdf')